In [1]:
import pyarrow.parquet as pq
import dask.dataframe as dd
import os

In [2]:
# Define paths
base_path_merge = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged"
base_path_filtered = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered"
output_dir = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged"

merged_file = os.path.join(base_path_merge, "merged_bookings_measurements_cleaned.parquet")
filtered_materials_file  = os.path.join(base_path_filtered, "filtered_materials_encoded.parquet")
final_output_path = os.path.join(output_dir, "final_merged_all.parquet")

## Load filtered parquet files

In [3]:
# Load filtered parquet files
materials_ddf = dd.read_parquet(filtered_materials_file)
merged_ddf = dd.read_parquet(merged_file)

## Merge Datasets

In [4]:
# Filter by created_at overlap (using Dask's lazy filtering)
start_time = max(materials_ddf['created_at'].min().compute(), merged_ddf['created_at'].min().compute())
end_time = min(materials_ddf['created_at'].max().compute(), merged_ddf['created_at'].max().compute())

filtered_materials_ddf = materials_ddf[(materials_ddf['created_at'] >= start_time) & (materials_ddf['created_at'] <= end_time)]
filtered_merged_ddf = merged_ddf[(merged_ddf['created_at'] >= start_time) & (merged_ddf['created_at'] <= end_time)]

In [5]:
# Get unique serial_number_ids from filtered_merged_ddf (compute to get Pandas array)
unique_serial_ids = filtered_merged_ddf['serial_number_id'].unique().compute()

chunk_size = 5_000  # TODO Tune this based on your memory capacity
num_chunks = (len(unique_serial_ids) + chunk_size - 1) // chunk_size

chunk_files = []

for i in range(num_chunks):
    chunk_ids = unique_serial_ids[i*chunk_size : (i+1)*chunk_size]

    # Filter both datasets on this chunk's serial_number_ids
    chunk_merged = filtered_merged_ddf[filtered_merged_ddf['serial_number_id'].isin(chunk_ids)]
    chunk_materials = filtered_materials_ddf[filtered_materials_ddf['serial_number_id'].isin(chunk_ids)]

    # Merge chunk
    chunk_merged_df = chunk_merged.merge(chunk_materials, on='serial_number_id', suffixes=('', '_mat'))

    # Drop created_at columns
    created_at_cols = [col for col in chunk_merged_df.columns if col == 'created_at' or col.endswith('created_at')]
    chunk_merged_df = chunk_merged_df.drop(columns=created_at_cols)

    # Remove duplicate columns with '_mat'
    mat_suffix_cols = [col for col in chunk_merged_df.columns if col.endswith('_mat')]
    for col in mat_suffix_cols:
        original_col = col[:-4]
        if original_col in chunk_merged_df.columns:
            chunk_merged_df = chunk_merged_df.drop(columns=[col])
        else:
            chunk_merged_df = chunk_merged_df.rename(columns={col: original_col})

    # Drop duplicate rows
    chunk_merged_df = chunk_merged_df.drop_duplicates()

    # Compute this chunk (bring into memory as Pandas)
    chunk_df = chunk_merged_df.compute()

    # Save this chunk to parquet
    chunk_file = os.path.join(output_dir, f"merged_chunk_{i}.parquet")
    chunk_df.to_parquet(chunk_file, index=False)
    chunk_files.append(chunk_file)

    print(f"Saved chunk {i+1}/{num_chunks} with shape {chunk_df.shape}")

Saved chunk 1/10 with shape (0, 23)
Saved chunk 2/10 with shape (26861149, 23)


MemoryError: Unable to allocate 817. MiB for an array with shape (3, 35704594) and data type float64